# Binary classification: in-hospital mortality
This workshop notebook assumes the 1,000 synthetic profiles are available at `D:\\CTSA\\synthetic_patient_profiles_1000.csv`. Synthetic educational data only; not for clinical decisions.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (roc_auc_score, average_precision_score, classification_report,
                             f1_score, precision_score, recall_score, balanced_accuracy_score,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

# Locate the repository root whether Jupyter was started in the repo root or in notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = REPO_ROOT / "data" / "profiles_1000"
DATA_PATH = DATA_ROOT / 'synthetic_patient_profiles_1000.csv'
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## Leakage-safe features and provided splits
The target is `in_hospital_mortality`. Disease class, length of stay, log length of stay, and ICU admission are excluded because they are outcomes, retrospective information, or potentially observed after the admission prediction time.

## Models compared
| Model | Strengths | Limitations |
|---|---|---|
| Logistic Regression | Fast, stable, and interpretable | Assumes a linear relationship in log-odds |
| Decision Tree | Easy to visualize; captures nonlinear rules | Can overfit and have high variance |
| Random Forest | Strong nonlinear baseline; reduces tree variance | Slower and less directly interpretable |
| SVM (RBF) | Effective in high-dimensional nonlinear settings | Probability fitting can be slow and memory intensive |
| LightGBM | Fast gradient boosting with strong tabular performance | Sensitive to tuning; less transparent |
| XGBoost | Accurate, flexible, and robust gradient boosting | More tuning and computation than simpler models |

In [ ]:
TARGET = 'in_hospital_mortality'
EXCLUDE = ['synthetic_patient_id', TARGET, 'primary_disease_class',
           'length_of_stay_days', 'log1p_length_of_stay', 'icu_admission']
task = pd.read_csv(DATA_ROOT / 'mortality_prediction_dataset.csv')
feature_cols = [c for c in task.columns if c not in {TARGET, 'split', 'synthetic_patient_id'}]
train = task[task['split'] == 'train']
valid = task[task['split'] == 'validation']
test = task[task['split'] == 'test']
X_train, y_train = train[feature_cols], train[TARGET]
X_valid, y_valid = valid[feature_cols], valid[TARGET]
X_test, y_test = test[feature_cols], test[TARGET]
print({s: len(x) for s, x in [('train', train), ('validation', valid), ('test', test)]})
print('Training mortality prevalence:', y_train.mean())

In [ ]:
numeric = X_train.select_dtypes(include='number').columns.tolist()
categorical = [c for c in feature_cols if c not in numeric]
preprocess = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')),
                          ('scaler', StandardScaler())]), numeric),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                              ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical),
])

candidate_classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=500, min_samples_leaf=3, class_weight='balanced', random_state=42, n_jobs=-1),
    'SVM': SVC(C=1.0, kernel='rbf', class_weight='balanced', probability=True, random_state=42),
}

# LightGBM and XGBoost are optional third-party packages.
try:
    from lightgbm import LGBMClassifier
    candidate_classifiers['LightGBM'] = LGBMClassifier(
        n_estimators=300, learning_rate=0.03, num_leaves=15, class_weight='balanced',
        random_state=42, n_jobs=-1, verbosity=-1)
except ImportError:
    print('LightGBM not installed; run: pip install lightgbm')

try:
    from xgboost import XGBClassifier
    negative, positive = np.bincount(y_train.astype(int))
    candidate_classifiers['XGBoost'] = XGBClassifier(
        n_estimators=300, learning_rate=0.03, max_depth=3, subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=negative / positive, eval_metric='logloss', random_state=42, n_jobs=-1)
except ImportError:
    print('XGBoost not installed; run: pip install xgboost')

models = {}
for name, classifier in candidate_classifiers.items():
    pipeline = Pipeline([('preprocess', clone(preprocess)), ('classifier', classifier)])
    pipeline.fit(X_train, y_train)
    models[name] = pipeline
    print(f'Fitted {name}')

In [ ]:
threshold_grid = np.linspace(0.01, 0.99, 99)
evaluation = {}
rows = []

for name, fitted_model in models.items():
    valid_probability = fitted_model.predict_proba(X_valid)[:, 1]
    valid_f1 = [f1_score(y_valid, valid_probability >= t, zero_division=0) for t in threshold_grid]
    threshold = float(threshold_grid[int(np.argmax(valid_f1))])
    test_probability = fitted_model.predict_proba(X_test)[:, 1]
    test_prediction = (test_probability >= threshold).astype(int)
    evaluation[name] = {'threshold': threshold, 'valid_probability': valid_probability,
                        'test_probability': test_probability, 'test_prediction': test_prediction}
    rows.append({
        'Model': name,
        'Validation AUROC': roc_auc_score(y_valid, valid_probability),
        'Threshold': threshold,
        'Test AUROC': roc_auc_score(y_test, test_probability),
        'Test AUPRC': average_precision_score(y_test, test_probability),
        'Balanced accuracy': balanced_accuracy_score(y_test, test_prediction),
        'Precision': precision_score(y_test, test_prediction, zero_division=0),
        'Recall': recall_score(y_test, test_prediction, zero_division=0),
        'F1': f1_score(y_test, test_prediction, zero_division=0),
    })

results = pd.DataFrame(rows).sort_values('Validation AUROC', ascending=False).reset_index(drop=True)
best_model_name = results.loc[0, 'Model']  # Selected without looking at test performance.
best_model = models[best_model_name]
print(f'Best model by validation AUROC: {best_model_name}')
display(results.style.format({c: '{:.3f}' for c in results.columns if c != 'Model'}))
print('\nTest classification report for the validation-selected model:')
print(classification_report(y_test, evaluation[best_model_name]['test_prediction'], zero_division=0))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for name, values in evaluation.items():
    probability = values['test_probability']
    RocCurveDisplay.from_predictions(y_test, probability, ax=axes[0, 0], name=name)
    PrecisionRecallDisplay.from_predictions(y_test, probability, ax=axes[0, 1], name=name)
    prob_true, prob_pred = calibration_curve(y_test, probability, n_bins=5, strategy='quantile')
    axes[1, 0].plot(prob_pred, prob_true, marker='o', label=name)

axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0, 0].set_title('ROC curves — all models')
axes[0, 1].set_title('Precision-recall curves — all models')
axes[1, 0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
axes[1, 0].set(xlabel='Mean predicted probability', ylabel='Observed event rate',
               title='Calibration plots — all models', xlim=(0, 1), ylim=(0, 1))
axes[1, 0].legend(fontsize=8)

best_values = evaluation[best_model_name]
ConfusionMatrixDisplay.from_predictions(
    y_test, best_values['test_prediction'], ax=axes[1, 1], colorbar=False)
axes[1, 1].set_title(f'{best_model_name} confusion matrix\nthreshold={best_values["threshold"]:.2f}')
fig.tight_layout()
plt.show()

## SHAP model explanations
SHAP values explain the model selected by validation AUROC on its preprocessed feature space. The model-agnostic probability explainer works across all six classifier types. The beeswarm shows effect direction, the bar chart ranks global importance, and the waterfall explains one high-risk test prediction. Install SHAP with pip install shap if needed.

In [ ]:
try:
    import shap
except ImportError as exc:
    raise ImportError('SHAP is required for these plots. Install it with: pip install shap') from exc

fitted_preprocess = best_model.named_steps['preprocess']
fitted_classifier = best_model.named_steps['classifier']
X_train_transformed = fitted_preprocess.transform(X_train)
X_test_transformed = fitted_preprocess.transform(X_test)
feature_names = fitted_preprocess.get_feature_names_out()

# Dense arrays make SHAP plotting reliable; this workshop dataset is small.
if hasattr(X_train_transformed, 'toarray'):
    X_train_transformed = X_train_transformed.toarray()
if hasattr(X_test_transformed, 'toarray'):
    X_test_transformed = X_test_transformed.toarray()

X_train_shap = pd.DataFrame(X_train_transformed, columns=feature_names)
X_test_shap = pd.DataFrame(X_test_transformed, columns=feature_names)
background = shap.sample(X_train_shap, min(100, len(X_train_shap)), random_state=42)
explain_rows = X_test_shap.iloc[:min(100, len(X_test_shap))]
explainer = shap.Explainer(fitted_classifier.predict_proba, background)
raw_shap_values = explainer(explain_rows)
# Binary predict_proba returns explanations for classes 0 and 1; retain class 1.
shap_values = raw_shap_values[:, :, 1] if raw_shap_values.values.ndim == 3 else raw_shap_values

print(f'SHAP explanations for: {best_model_name}')
shap.plots.beeswarm(shap_values, max_display=15)
shap.plots.bar(shap_values, max_display=15)

explained_probability = evaluation[best_model_name]['test_probability'][:len(explain_rows)]
example_index = int(np.argmax(explained_probability))
print(f'Waterfall for test row {example_index}; predicted risk={explained_probability[example_index]:.3f}')
shap.plots.waterfall(shap_values[example_index], max_display=15)